# Short-Expiry Risk Dashboard

This notebook groups option contracts by days to expiry and compares Gamma, Theta, Vega, bid-ask spread, and implied-volatility uncertainty. The goal is to show how short-dated option risk can behave differently from longer-dated option risk.

## DTE buckets

DTE means days to expiry. The dashboard uses these buckets:

| Bucket | Meaning |
|---|---|
| `0_to_1_DTE` | Same-day or next-day expiry |
| `2_to_7_DTE` | Very short weekly expiry |
| `8_to_14_DTE` | One to two weeks |
| `15_to_30_DTE` | Two weeks to one month |
| `31_to_60_DTE` | One to two months |

Short-dated options can have high Gamma, fast time decay, and quote-quality issues that make IV and Greek estimates less stable.

In [ ]:
from pathlib import Path
import sys

import pandas as pd

In [ ]:
project_root = Path.cwd()

if not (project_root / "src").exists():
    project_root = project_root.parent

sys.path.insert(0, str(project_root))

In [ ]:
from src.short_expiry_risk import (
    build_short_expiry_dashboard_outputs,
    prepare_short_expiry_dataset,
    short_expiry_risk_score,
    short_expiry_risk_summary,
)

## Load Greek uncertainty data

The input should contain mid Greeks, IV uncertainty, spread percentage, and days to expiry.

In [ ]:
input_path = project_root / "data" / "processed" / "greek_uncertainty_results.csv"

if not input_path.exists():
    raise FileNotFoundError("Greek uncertainty dataset not found.")

greek_data = pd.read_csv(input_path)
greek_data.head(10)

## Prepare short-expiry dataset

In [ ]:
short_expiry_data = prepare_short_expiry_dataset(
    greek_data,
    max_days=60,
    retained_only=True,
)

short_expiry_data = short_expiry_risk_score(short_expiry_data)

short_expiry_data.head(10)

## Short-expiry risk summary

In [ ]:
summary = short_expiry_risk_summary(short_expiry_data)
summary

In [ ]:
processed_dir = project_root / "data" / "processed"
processed_dir.mkdir(exist_ok=True)

summary_path = processed_dir / "short_expiry_risk_summary.csv"
short_expiry_data_path = processed_dir / "short_expiry_risk_dataset.csv"

summary.to_csv(summary_path, index=False)
short_expiry_data.to_csv(short_expiry_data_path, index=False)

summary_path, short_expiry_data_path

## Dashboard figures

In [ ]:
figures_dir = project_root / "figures"
figures_dir.mkdir(exist_ok=True)

figure_paths = build_short_expiry_dashboard_outputs(
    short_expiry_data,
    output_dir=figures_dir,
)

figure_paths

## Interpretation

Short-expiry risk is not only about higher Gamma or faster Theta. Quote quality also matters. When bid-ask spreads and IV uncertainty are high, the Greeks used for hedging can become less reliable. This connects expiry risk directly to the project's market-friction theme.